# Post-Processing Ablation Study

This notebook measures the **individual contribution** of each post-processing step to road segmentation quality. We start from raw model predictions and incrementally add steps, measuring IoU/Dice at each stage.

### Steps tested (incremental)
| # | Step | What it does |
|---|------|-------------|
| 0 | Raw threshold (0.5) | Baseline — no post-processing |
| 1 | Optimal threshold | Sweep 0.20-0.80, pick best IoU |
| 2 | + Component filter | Remove small isolated blobs |
| 3 | + Morphological close | Fill tiny gaps in roads |
| 4 | + Morphological open | Remove thin noise |
| 5 | + Gap bridging | Connect dead-ends using probability map |
| 6 | TTA (independent) | 6-fold test-time augmentation |

---
## 1. Setup

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
    PROJECT_ROOT = __import__('pathlib').Path(REPO_DIR).resolve()
else:
    PROJECT_ROOT = __import__('pathlib').Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

print(f"Project root: {PROJECT_ROOT}")

---
## 2. Load Model and Generate Predictions

We need raw probability maps (not binary masks) to test post-processing.

In [ ]:
from pathlib import Path
import torch
import numpy as np
from tqdm import tqdm

# ====== SET YOUR CHECKPOINT PATH ======
# Local:
# CHECKPOINT = "checkpoints/Unet_resnet34_20260406_045917/best.pth"
# From Drive:
# CHECKPOINT = "/content/drive/MyDrive/road_segmentation/checkpoints/.../best.pth"
CHECKPOINT = ""  # <-- SET THIS

assert CHECKPOINT, "Set CHECKPOINT path above"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load model from checkpoint
from road_segmentation.api.inference import InferenceEngine
state = torch.load(CHECKPOINT, map_location=device, weights_only=False)
config_dict = state.get("config", {})
model_cfg = config_dict.get("model", {})
data_cfg = config_dict.get("data", {})

from road_segmentation.models.factory import create_model
model = create_model(
    arch=model_cfg.get("arch", "Unet"),
    encoder_name=model_cfg.get("encoder_name", "resnet34"),
    encoder_weights=None,
    in_channels=model_cfg.get("in_channels", 3),
    classes=model_cfg.get("classes", 1),
)
model.load_state_dict(state["model_state_dict"])
model = model.to(device).eval()

image_size = data_cfg.get("image_size", 512)
print(f"Model: {model_cfg.get('arch')}+{model_cfg.get('encoder_name')} | image_size={image_size}")

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR
from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs
from road_segmentation.data.transforms import get_val_transform
from road_segmentation.data.dataset import RoadSegmentationDataset
from torch.utils.data import DataLoader

norm_cfg = config_dict.get("normalization", {})
mean = norm_cfg.get("mean", [0.485, 0.456, 0.406])
std = norm_cfg.get("std", [0.229, 0.224, 0.225])

pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
_, val_pairs = split_pairs(
    pairs,
    val_ratio=config_dict.get("data", {}).get("val_split_ratio", 0.15),
    seed=config_dict.get("data", {}).get("val_split_seed", 42),
)

val_transform = get_val_transform(image_size, mean, std)
val_dataset = RoadSegmentationDataset(val_pairs, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"Validation set: {len(val_pairs)} samples")
print(f"Generating probability maps...")

# Generate all probability maps and ground truths
all_probs = []
all_gts = []

with torch.no_grad():
    for batch in tqdm(val_loader):
        images = batch["image"].to(device)
        masks = batch["mask"]  # (N, 1, H, W)

        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()

        for i in range(len(probs)):
            all_probs.append(probs[i, 0])  # (H, W)
            all_gts.append(masks[i, 0].numpy())  # (H, W)

print(f"Generated {len(all_probs)} probability maps")

---
## 3. Threshold Optimization

Find the threshold that maximizes IoU on the validation set.

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("ggplot")

from road_segmentation.postprocessing.steps import find_optimal_threshold

best_threshold, threshold_results = find_optimal_threshold(
    all_probs, all_gts,
    thresholds=np.arange(0.15, 0.85, 0.05),
)

print(f"Optimal threshold: {best_threshold}")
print(f"IoU at 0.50: {threshold_results.get(0.5, 'N/A')}")
print(f"IoU at optimal: {threshold_results[best_threshold]}")
print(f"Improvement: +{threshold_results[best_threshold] - threshold_results.get(0.5, 0):.4f}")

# Plot threshold curve
fig, ax = plt.subplots(figsize=(10, 5))
thresholds = sorted(threshold_results.keys())
ious = [threshold_results[t] for t in thresholds]
ax.plot(thresholds, ious, 'o-', linewidth=2, markersize=6)
ax.axvline(best_threshold, color='red', linestyle='--', label=f'Best: {best_threshold} (IoU={threshold_results[best_threshold]:.4f})')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label=f'Default 0.5 (IoU={threshold_results.get(0.5, 0):.4f})')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Validation IoU', fontsize=12)
ax.set_title('Threshold Sweep', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True)
plt.tight_layout()
plt.show()

---
## 4. Ablation Study — Step-by-Step

Apply each post-processing step incrementally and measure its contribution.

In [ ]:
import pandas as pd
from road_segmentation.postprocessing.pipeline import PipelineConfig, run_pipeline, ablation_configs


def compute_metrics(masks_pred, masks_gt):
    """Compute micro-averaged IoU, Dice, Precision, Recall."""
    tp, fp, fn = 0, 0, 0
    for pred, gt in zip(masks_pred, masks_gt):
        p = (pred > 0).astype(bool)
        g = (gt > 0).astype(bool)
        tp += int(np.sum(p & g))
        fp += int(np.sum(p & ~g))
        fn += int(np.sum(~p & g))
    iou = tp / max(tp + fp + fn, 1)
    dice = 2 * tp / max(2 * tp + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return {"IoU": iou, "Dice": dice, "Precision": precision, "Recall": recall}


# Get ablation configs and set optimal threshold
configs = ablation_configs()
for name, cfg in configs.items():
    if name != "00_raw_threshold_0.5":
        cfg.threshold = best_threshold

# Run each config on all validation samples
ablation_results = {}

for config_name, pipeline_cfg in configs.items():
    print(f"Running: {config_name}...")
    masks_pred = []
    for prob in all_probs:
        result = run_pipeline(prob, pipeline_cfg)
        masks_pred.append(result.mask)

    metrics = compute_metrics(masks_pred, all_gts)
    ablation_results[config_name] = metrics
    print(f"  IoU={metrics['IoU']:.4f} | Dice={metrics['Dice']:.4f}")

print("\nDone.")

### TTA Evaluation (independent test)

Test-time augmentation is applied **before** thresholding, so it's evaluated separately.

In [ ]:
from road_segmentation.postprocessing.steps import apply_tta

# Run TTA on a subset (it's 6x slower)
N_TTA_SAMPLES = min(100, len(val_pairs))
print(f"Running TTA on {N_TTA_SAMPLES} samples (6-fold)...")

tta_probs = []
tta_gts = all_gts[:N_TTA_SAMPLES]

for i in tqdm(range(N_TTA_SAMPLES)):
    sample = val_dataset[i]
    image_tensor = sample["image"].unsqueeze(0)  # (1, C, H, W)
    tta_prob = apply_tta(model, image_tensor, device)
    tta_probs.append(tta_prob)

# Compare: raw vs TTA (both at optimal threshold)
raw_masks = [(p >= best_threshold).astype(np.uint8) * 255 for p in all_probs[:N_TTA_SAMPLES]]
tta_masks = [(p >= best_threshold).astype(np.uint8) * 255 for p in tta_probs]

raw_metrics = compute_metrics(raw_masks, tta_gts)
tta_metrics = compute_metrics(tta_masks, tta_gts)

print(f"\nWithout TTA: IoU={raw_metrics['IoU']:.4f} | Dice={raw_metrics['Dice']:.4f}")
print(f"With TTA:    IoU={tta_metrics['IoU']:.4f} | Dice={tta_metrics['Dice']:.4f}")
print(f"TTA gain:    +{tta_metrics['IoU'] - raw_metrics['IoU']:.4f} IoU")

---
## 5. Results Dashboard

In [ ]:
# Build comparison table
rows = []
prev_iou = 0
for name, metrics in ablation_results.items():
    delta = metrics["IoU"] - prev_iou
    rows.append({
        "Step": name,
        "IoU": metrics["IoU"],
        "Dice": metrics["Dice"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "Delta IoU": f"+{delta:.4f}" if delta > 0 else f"{delta:.4f}",
    })
    prev_iou = metrics["IoU"]

# Add TTA row
rows.append({
    "Step": "TTA (6-fold, independent)",
    "IoU": tta_metrics["IoU"],
    "Dice": tta_metrics["Dice"],
    "Precision": tta_metrics["Precision"],
    "Recall": tta_metrics["Recall"],
    "Delta IoU": f"+{tta_metrics['IoU'] - raw_metrics['IoU']:.4f}",
})

results_df = pd.DataFrame(rows)
print("Post-Processing Ablation Results")
print("=" * 80)
display(results_df.round(4))

In [ ]:
# Bar chart of IoU per step
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# IoU progression
step_names = [r["Step"].split("_", 1)[-1] if "_" in r["Step"] else r["Step"] for r in rows[:-1]]
step_ious = [r["IoU"] for r in rows[:-1]]

colors = ['#2b8cbe' if i == 0 else '#41ae76' if step_ious[i] > step_ious[i-1] else '#ef6548'
          for i in range(len(step_ious))]

axes[0].barh(range(len(step_names)), step_ious, color=colors, edgecolor='white')
axes[0].set_yticks(range(len(step_names)))
axes[0].set_yticklabels(step_names, fontsize=10)
axes[0].set_xlabel('IoU', fontsize=12)
axes[0].set_title('IoU After Each Post-Processing Step', fontsize=13)
axes[0].invert_yaxis()
for i, v in enumerate(step_ious):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

# Delta IoU
deltas = [0] + [step_ious[i] - step_ious[i-1] for i in range(1, len(step_ious))]
delta_colors = ['#41ae76' if d >= 0 else '#ef6548' for d in deltas]
axes[1].barh(range(len(step_names)), deltas, color=delta_colors, edgecolor='white')
axes[1].set_yticks(range(len(step_names)))
axes[1].set_yticklabels(step_names, fontsize=10)
axes[1].set_xlabel('Delta IoU', fontsize=12)
axes[1].set_title('Contribution of Each Step', fontsize=13)
axes[1].axvline(0, color='black', linewidth=0.5)
axes[1].invert_yaxis()
for i, v in enumerate(deltas):
    axes[1].text(v + 0.001 if v >= 0 else v - 0.008, i, f'{v:+.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 6. Visual Comparison

Show the same samples before and after each step.

In [ ]:
from road_segmentation.training.visualization import denormalize_image

# Run full pipeline with intermediates on a few samples
full_config = PipelineConfig(
    threshold=best_threshold,
    remove_small_components=True,
    min_component_area=100,
    morphological_closing=True,
    closing_kernel_size=5,
    morphological_opening=True,
    opening_kernel_size=3,
    gap_bridging=True,
    max_gap_px=30,
    min_bridge_confidence=0.3,
)

N_VIS = 4
vis_indices = np.linspace(0, len(all_probs) - 1, N_VIS, dtype=int)

for idx in vis_indices:
    result = run_pipeline(all_probs[idx], full_config, collect_intermediates=True)

    # Get original image
    sample = val_dataset[idx]
    img = denormalize_image(sample["image"], mean, std)
    gt = all_gts[idx]

    intermediates = result.intermediates
    step_names_vis = list(intermediates.keys())
    n_cols = 3 + len(step_names_vis)  # image, GT, prob, then each step

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    axes[0].imshow(img)
    axes[0].set_title("Input", fontsize=10)
    axes[0].axis("off")

    axes[1].imshow(gt, cmap="gray")
    axes[1].set_title("Ground Truth", fontsize=10)
    axes[1].axis("off")

    axes[2].imshow(all_probs[idx], cmap="hot", vmin=0, vmax=1)
    axes[2].set_title("Probability Map", fontsize=10)
    axes[2].axis("off")

    for j, (step_name, step_mask) in enumerate(intermediates.items()):
        axes[3 + j].imshow(step_mask, cmap="gray")
        short_name = step_name.split("_", 1)[-1]
        axes[3 + j].set_title(short_name, fontsize=9)
        axes[3 + j].axis("off")

    plt.suptitle(f"Sample {idx}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## 7. Decision Summary

Based on the ablation results, select which steps to include in the final pipeline.

In [ ]:
print("Decision Guide")
print("=" * 60)
print()

for i, row in enumerate(rows):
    delta = float(row["Delta IoU"].replace("+", "")) if isinstance(row["Delta IoU"], str) else row["Delta IoU"]
    if i == 0:
        verdict = "BASELINE"
    elif delta > 0.005:
        verdict = "INCLUDE (significant improvement)"
    elif delta > 0.001:
        verdict = "OPTIONAL (marginal improvement)"
    elif delta > -0.001:
        verdict = "NEUTRAL (no meaningful change)"
    else:
        verdict = "SKIP (hurts performance)"

    step_name = row["Step"].split("_", 1)[-1] if "_" in row["Step"] else row["Step"]
    print(f"{step_name:30s} | IoU={row['IoU']:.4f} | {row['Delta IoU']:>8s} | {verdict}")